In [33]:
from gbp import gbp
from gbp.factors import linear_displacement
import numpy as np

np.random.seed(0)

In [34]:
class args:
    dim = 2
    M = 10
    n_varnodes = 50
    gauss_noise_std = 1.
    n_iters = 100

In [35]:
# Create priors
priors_mu = np.random.rand(args.n_varnodes, args.dim) * 10  # grid goes from 0 to 10 along x and y axis
prior_sigma = 3 * np.eye(args.dim)

prior_lambda = np.linalg.inv(prior_sigma)
priors_lambda = [prior_lambda] * args.n_varnodes
priors_eta = []
for mu in priors_mu:
    priors_eta.append(np.dot(prior_lambda, mu))

print(priors_mu)
print(priors_eta)

# Generate connections between variables
gt_measurements, noisy_measurements = [], []
measurements_nodeIDs = []
num_edges_per_node = np.zeros(args.n_varnodes)
n_edges = 0

for i, mu in enumerate(priors_mu):
    dists = []
    for j, mu1 in enumerate(priors_mu):
        dists.append(np.linalg.norm(mu - mu1))
    for j in np.array(dists).argsort()[1:args.M + 1]:  # As closest node is itself
        mu1 = priors_mu[j]
        if [j, i] not in measurements_nodeIDs:  # To avoid double counting
            n_edges += 1
            gt_measurements.append(mu - mu1)
            noisy_measurements.append(mu - mu1 + np.random.normal(0., args.gauss_noise_std, args.dim))
            measurements_nodeIDs.append([i, j])

            num_edges_per_node[i] += 1
            num_edges_per_node[j] += 1

graph = gbp.FactorGraph(nonlinear_factors=False)

# Initialize variable nodes for frames with prior
for i in range(args.n_varnodes):
    new_var_node = gbp.VariableNode(i, args.dim)
    new_var_node.prior.eta = priors_eta[i]
    new_var_node.prior.lam = priors_lambda[i]
    graph.var_nodes.append(new_var_node)

for f, measurement in enumerate(noisy_measurements):
    new_factor = gbp.Factor(f,
                            [graph.var_nodes[measurements_nodeIDs[f][0]], graph.var_nodes[measurements_nodeIDs[f][1]]],
                            measurement,
                            args.gauss_noise_std,
                            linear_displacement.meas_fn,
                            linear_displacement.jac_fn,
                            loss=None,
                            mahalanobis_threshold=2)

    graph.var_nodes[measurements_nodeIDs[f][0]].adj_factors.append(new_factor)
    graph.var_nodes[measurements_nodeIDs[f][1]].adj_factors.append(new_factor)
    graph.factors.append(new_factor)

graph.update_all_beliefs()
graph.compute_all_factors()

graph.n_var_nodes = args.n_varnodes
graph.n_factor_nodes = len(noisy_measurements)
graph.n_edges = 2 * len(noisy_measurements)

print(f'Number of variable nodes {graph.n_var_nodes}')
print(f'Number of edges per variable node {args.M}')
print(f'Number of dofs at each variable node {args.dim}\n')

mu, sigma = graph.joint_distribution_cov()  # Get batch solution

[[5.48813504 7.15189366]
 [6.02763376 5.44883183]
 [4.23654799 6.45894113]
 [4.37587211 8.91773001]
 [9.63662761 3.83441519]
 [7.91725038 5.2889492 ]
 [5.68044561 9.25596638]
 [0.71036058 0.871293  ]
 [0.20218397 8.32619846]
 [7.78156751 8.70012148]
 [9.78618342 7.99158564]
 [4.61479362 7.80529176]
 [1.18274426 6.39921021]
 [1.43353287 9.44668917]
 [5.21848322 4.1466194 ]
 [2.64555612 7.74233689]
 [4.56150332 5.68433949]
 [0.187898   6.17635497]
 [6.12095723 6.16933997]
 [9.43748079 6.81820299]
 [3.59507901 4.37031954]
 [6.97631196 0.60225472]
 [6.66766715 6.7063787 ]
 [2.10382561 1.28926298]
 [3.15428351 3.63710771]
 [5.7019677  4.38601513]
 [9.88373838 1.02044811]
 [2.08876756 1.61309518]
 [6.53108325 2.53291603]
 [4.66310773 2.44425592]
 [1.58969584 1.10375141]
 [6.56329589 1.38182951]
 [1.96582362 3.68725171]
 [8.2099323  0.97101276]
 [8.37944907 0.96098408]
 [9.76459465 4.68651202]
 [9.76761088 6.0484552 ]
 [7.39263579 0.39187792]
 [2.82806963 1.20196561]
 [2.96140198 1.18727719]


In [36]:
import pygame
import random
import sys

# Initialize Pygame
pygame.init()

# Screen dimensions
screen_width = 1000
screen_height = 1000
screen = pygame.display.set_mode((screen_width, screen_height))

# Title and Icon
pygame.display.set_caption("Moving Points with Transparent Circles")

# Color definitions
black = (0, 0, 0)
green = (0, 255, 0, 128)  # Green with half transparency
white = (255, 255, 255)
blue = (0, 0, 255)
red = (255, 0, 0)


# Function to draw a semi-transparent circle around a point
def draw_transparent_circle(surface, color, center, radius):
    # Create a temporary surface with per-pixel alpha
    temp_surface = pygame.Surface((2 * radius, 2 * radius), pygame.SRCALPHA)
    pygame.draw.circle(temp_surface, color, (radius, radius), radius)
    surface.blit(temp_surface, (center[0] - radius, center[1] - radius))


# Function to draw a semi-transparent ellipse around a point
def draw_transparent_ellipse(surface, color, center, size):
    # Create a temporary surface with per-pixel alpha
    temp_surface = pygame.Surface(size, pygame.SRCALPHA)
    # The ellipse will be drawn inside a rect defined by the size, so the position (0,0) is relative to the surface
    pygame.draw.ellipse(temp_surface, color, (0, 0, size[0], size[1]))
    # Blit this surface onto the main surface, centered on the given coordinates
    surface.blit(temp_surface, (center[0] - size[0] // 2, center[1] - size[1] // 2))


def draw_transparent_line(surface, color, start, end):
    # Create a temporary surface with per-pixel alpha, large enough to contain the line
    # For simplicity, this example uses the size of the entire screen, but you could use a smaller surface if preferred
    temp_surface = pygame.Surface(surface.get_size(), pygame.SRCALPHA)

    # Draw the line on the temporary surface
    pygame.draw.line(temp_surface, color, start, end, 1)

    # Blit this surface onto the main surface
    surface.blit(temp_surface, (0, 0))


def get_line_of_factor(factor):
    mu1 = factor.adj_var_nodes[0].mu
    mu2 = factor.adj_var_nodes[1].mu
    return mu1, mu2


# Function to wait for space bar
def wait_for_space():
    waiting = True
    while waiting:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_SPACE:
                    waiting = False


# Main loop
running = True

means, sigmas = graph.joint_distribution_cov()
print(means)
print(sigmas)

iter = 0
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Fill the screen with black
    screen.fill(black)

    if iter < args.n_iters:
        graph.synchronous_iteration()
    iter += 1

    mu_estimates = []

    # Move and draw points
    for i in range(args.n_varnodes):
        pos = graph.var_nodes[i].mu
        mu_estimates.append(pos)
        mu = (int(pos[0] * 100), int(pos[1] * 100))
        # sigma = sigmas[i]
        # take diagonal of covariance matrix
        # sigma_diag = np.diag(sigma)
        # Draw the semi-transparent circle around the point
        # draw_transparent_ellipse(screen, green, mu, (int(sigma_diag[0] * 100), int(sigma_diag[1] * 100)))

        # Draw the point itself
        pygame.draw.circle(screen, blue, mu, 3)

        # Draw the ground truth
        mu_gt = (int(means[i * 2] * 100), int(means[i * 2 + 1] * 100))
        print(mu, mu_gt)
        pygame.draw.circle(screen, white, mu_gt, 3)

        draw_transparent_line(screen, red, mu, mu_gt)

    # Draw the factors
    for factor in graph.factors:
        mu1, mu2 = get_line_of_factor(factor)
        mu1 = (int(mu1[0] * 100), int(mu1[1] * 100))
        mu2 = (int(mu2[0] * 100), int(mu2[1] * 100))
        draw_transparent_line(screen, white, mu1, mu2)

    # wait_for_space()

    # Update the display
    pygame.display.flip()

    # Cap the frame rate
    pygame.time.Clock().tick(10)

# Quit Pygame
pygame.quit()


[4.82118119 2.29053639 4.23811222 3.66436316 5.50766973 3.51289432
 5.4489025  1.40270799 1.81659553 4.87699145 2.98900685 4.6646945
 4.5780132  0.98218949 7.7204087  7.18864117 9.28445407 2.11114125
 2.71874478 1.91461182 1.27390876 2.26376027 5.39429489 2.24713165
 7.75950966 3.57559135 8.01042816 1.07490292 5.35782364 4.26637188
 6.83359226 3.06982244 5.12673556 4.09728081 8.57470624 3.90013922
 4.15420484 3.13726014 1.53049937 3.05552747 5.93914943 4.68686488
 4.01269917 6.2872735  3.68464643 3.66652039 6.74380048 6.58074243
 5.74424887 4.51703675 4.57564664 4.06581367 1.42465287 6.38597375
 6.36492149 6.5530125  4.23098288 5.50715183 5.31461839 5.15442851
 7.83652592 6.42605514 4.32469206 6.01606512 6.75002703 4.89740113
 2.44071166 6.33020305 2.64811247 6.43266054 1.78777081 5.06569666
 0.74764771 3.58637351 3.89935719 6.92594984 6.1416057  6.66714203
 5.80969747 6.47418614 5.96943824 4.4482493  8.18773106 3.05062337
 4.38180388 4.96472274 5.17113797 6.58033999 4.21955726 1.49467